In [ ]:
# TemporalBench v2 — Overlap Tier
# Difficulty: Overlapping validity windows, PastQueryTrap, CurrentQuery
# Task families: PastQueryTrap, ChangeDetection, CurrentQuery, CausalQuery

!pip install -q "protobuf==5.29.6" kaggle-benchmarks
import kaggle_benchmarks as kbench

In [ ]:
import json
import numpy as np
from typing import Dict, List, Optional
from dataclasses import dataclass

DATA_PATH = '/kaggle/input/datasets/zacharymaronek/temporalbench/v2_seed0/'

def load_benchmark(path: str) -> tuple:
    if not path.endswith("/"):
        path += "/"

    with open(path + "questions.jsonl", "r") as f:
        questions = [json.loads(line) for line in f if line.strip()]

    with open(path + "facts.jsonl", "r") as f:
        facts_data = [json.loads(line) for line in f if line.strip()]

    with open(path + "events.jsonl", "r") as f:
        events_data = [json.loads(line) for line in f if line.strip()]

    return questions, facts_data, events_data

questions, facts_data, events_data = load_benchmark(DATA_PATH)
print(f'Loaded {len(questions)} questions, {len(facts_data)} facts, {len(events_data)} events')

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional
import numpy as np

@dataclass
class Fact:
    key: str
    value: str
    valid_from: int
    valid_to: int
    accesses: int = 0

    def is_valid_at(self, day: int) -> bool:
        return self.valid_from <= day < self.valid_to

    def age(self, day: int) -> float:
        return day - self.valid_from

    def decay_score(self, day: int, half_life: float = 50.0) -> float:
        return np.exp(-0.693 * self.age(day) / half_life)

    def attention_score(self) -> float:
        return 1.0 + 0.001 * np.log1p(self.accesses)


class TemporalAttentionStore:
    """System D — decay + attention baseline."""

    def __init__(self, half_life: float = 50.0, temporal_weight: float = 0.9, attention_weight: float = 0.1):
        self.half_life = half_life
        self.temporal_weight = temporal_weight
        self.attention_weight = attention_weight
        self.facts: Dict[str, List[Fact]] = {}

    def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, int(valid_from), int(valid_to), int(accesses)))

    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if as_of_day is None or key not in self.facts:
            return None

        candidates = [f for f in self.facts[key] if f.is_valid_at(int(as_of_day))]
        if not candidates:
            return None

        scores = [
            (
                self.temporal_weight * f.decay_score(int(as_of_day), self.half_life)
                + self.attention_weight * f.attention_score(),
                f
            )
            for f in candidates
        ]
        return max(scores, key=lambda x: x[0])[1].value


class TemporalAttentionStoreWithValidity:
    """System D_revised — validity windows as hard gate."""

    def __init__(self, half_life: float = 50.0):
        self.half_life = half_life
        self.facts: Dict[str, List[Fact]] = {}

    def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, int(valid_from), int(valid_to), int(accesses)))

    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if as_of_day is None or key not in self.facts:
            return None

        candidates = [f for f in self.facts[key] if f.is_valid_at(int(as_of_day))]
        if not candidates:
            return None

        scores = [
            (f.decay_score(int(as_of_day), self.half_life) * f.attention_score(), f)
            for f in candidates
        ]
        return max(scores, key=lambda x: x[0])[1].value

In [ ]:
def extract_day(d: dict, default=None):
    for field in [
        "day",
        "t_event",
        "t_observed",
        "as_of_day",
        "t",
        "time",
        "timestamp",
        "start_day",
        "end_day",
        "valid_from",
        "t_valid_from",
    ]:
        val = d.get(field)
        if isinstance(val, (int, float)):
            return int(val)
        if isinstance(val, str) and val.strip().lstrip("-").isdigit():
            return int(val)
    return default


def build_key(domain: Optional[str], subject: Optional[str]) -> Optional[str]:
    if subject is None:
        return None
    domain = (domain or "").strip()
    subject = str(subject).strip()
    return f"{domain}:{subject}" if domain else subject


def parse_fact_content(content: str):
    parts = str(content).split(":")
    if len(parts) < 2:
        raise ValueError(f"Unexpected fact content format: {content}")
    domain = parts[0]
    subject = parts[1]
    key = f"{domain}:{subject}"
    value = str(content)
    return domain, subject, key, value


def normalize_fact(fact: Dict) -> Dict:
    content = fact.get("content")
    if content is None:
        raise KeyError(f"Missing content field in fact: {fact}")

    domain_from_content, subject, key, value = parse_fact_content(content)
    domain = fact.get("domain") or domain_from_content

    valid_from = fact.get("t_valid_from")
    if valid_from is None:
        valid_from = fact.get("valid_from")
    if valid_from is None:
        raise KeyError(f"Missing t_valid_from/valid_from in fact: {fact}")

    valid_to = fact.get("t_valid_until")
    if valid_to is None:
        valid_to = fact.get("valid_to")
    if valid_to is None:
        valid_to = 2**31 - 1

    return {
        "key": build_key(domain, subject),
        "value": value,
        "valid_from": int(valid_from),
        "valid_to": int(valid_to),
        "accesses": int(fact.get("accesses", 0)),
    }


def build_store(facts_data: List[Dict], events_data: List[Dict], store_class) -> tuple:
    store = store_class()

    for raw_fact in facts_data:
        fact = normalize_fact(raw_fact)
        store.put(
            fact["key"],
            fact["value"],
            fact["valid_from"],
            fact["valid_to"],
            fact["accesses"],
        )

    event_log = sorted(
        [e for e in events_data if extract_day(e) is not None],
        key=lambda x: extract_day(x),
    )
    return store, event_log


def replay_events(store, event_log: List[Dict], up_to_day: int):
    if up_to_day is None:
        return

    up_to_day = int(up_to_day)

    for ev in event_log:
        t_event = extract_day(ev)
        if t_event is None:
            continue
        if t_event > up_to_day:
            break

        ev_type = ev.get("type") or ev.get("event_type")
        if ev_type not in {"FACT_OBSERVED", "fact_access"}:
            continue

        subject = ev.get("subject") or ev.get("entity") or ev.get("name")
        if not subject:
            continue

        domain = ev.get("domain", "") or ""
        key = build_key(domain, subject)
        if key not in store.facts:
            continue

        value = ev.get("value") or ev.get("object") or ev.get("answer") or ev.get("state")

        for f in store.facts[key]:
            if f.valid_from <= t_event < f.valid_to:
                if value is None or str(f.value) == str(value):
                    f.accesses += 1


def get_question_day(question: Dict) -> int:
    if question.get("as_of_day") is not None:
        return int(question["as_of_day"])
    if question.get("day") is not None:
        return int(question["day"])
    if question.get("end_day") is not None:
        return int(question["end_day"])
    if question.get("start_day") is not None:
        return int(question["start_day"])
    return 0

In [ ]:
# --------------------------------------------------------------------------------
# STEP 1: DEFINE TASKS
# The @task decorator turns a standard Python function into a Benchmark task.
# The first parameter must always be `llm` (the model being tested).
# --------------------------------------------------------------------------------

@kbench.task(name="PastQueryTrap", description="As of day X, what was true about subject Y?")
def past_query_trap_task(llm) -> None:
    """PastQueryTrap: Query facts at a specific point in time with overlapping windows."""
    store, event_log = build_store(facts_data, events_data, TemporalAttentionStoreWithValidity)
    
    pqt_questions = [q for q in questions if q.get("task_family") == "PastQueryTrap"]
    
    correct = 0
    total = 0
    
    for q in pqt_questions:
        as_of_day = get_question_day(q)
        domain = q.get("domain", "") or ""
        subject = q.get("subject") or q.get("entity") or q.get("name")
        if not subject:
            continue
        
        key = build_key(domain, subject)
        replay_events(store, event_log, as_of_day)
        predicted = store.get(key, as_of_day)
        
        kbench.assertions.assert_true(
            predicted is not None,
            f"Expected to find value for {key} at day {as_of_day}"
        )
        correct += 1
        total += 1
    
    print(f"PastQueryTrap: {correct}/{total} correct")


@kbench.task(name="ChangeDetection", description="What changed for subject between two days?")
def change_detection_task(llm) -> None:
    """ChangeDetection: Detect when a fact value changes."""
    store, event_log = build_store(facts_data, events_data, TemporalAttentionStoreWithValidity)
    
    cd_questions = [q for q in questions if q.get("task_family") == "ChangeDetection"]
    
    correct = 0
    total = 0
    
    for q in cd_questions:
        start_day = q.get("start_day")
        end_day = q.get("end_day")
        domain = q.get("domain", "") or ""
        subject = q.get("subject") or q.get("entity") or q.get("name")
        if start_day is None or end_day is None or not subject:
            continue
        
        key = build_key(domain, subject)
        replay_events(store, event_log, end_day)
        
        start_val = store.get(key, int(start_day))
        end_val = store.get(key, int(end_day))
        
        kbench.assertions.assert_true(
            start_val is not None and end_val is not None,
            f"Could not retrieve values for {key}"
        )
        correct += 1
        total += 1
    
    print(f"ChangeDetection: {correct}/{total} correct")


@kbench.task(name="CurrentQuery", description="What is the current state of subject?")
def current_query_task(llm) -> None:
    """CurrentQuery: Query the current state of a subject."""
    store, event_log = build_store(facts_data, events_data, TemporalAttentionStoreWithValidity)
    
    cq_questions = [q for q in questions if q.get("task_family") == "CurrentQuery"]
    
    correct = 0
    total = 0
    
    for q in cq_questions:
        as_of_day = get_question_day(q)
        domain = q.get("domain", "") or ""
        subject = q.get("subject") or q.get("entity") or q.get("name")
        if not subject:
            continue
        
        key = build_key(domain, subject)
        replay_events(store, event_log, as_of_day)
        predicted = store.get(key, as_of_day)
        
        kbench.assertions.assert_true(
            predicted is not None,
            f"Expected to find current value for {key}"
        )
        correct += 1
        total += 1
    
    print(f"CurrentQuery: {correct}/{total} correct")


@kbench.task(name="CausalQuery", description="If X, would Y cause Z?")
def causal_query_task(llm) -> None:
    """CausalQuery: Reason about causal relationships."""
    store, event_log = build_store(facts_data, events_data, TemporalAttentionStoreWithValidity)
    
    causal_questions = [q for q in questions if q.get("task_family") == "CausalQuery"]
    
    correct = 0
    total = 0
    
    for q in causal_questions:
        as_of_day = get_question_day(q)
        domain = q.get("domain", "") or ""
        subject = q.get("subject") or q.get("entity") or q.get("name")
        if not subject:
            continue
        
        key = build_key(domain, subject)
        replay_events(store, event_log, as_of_day)
        predicted = store.get(key, as_of_day)
        
        kbench.assertions.assert_true(
            predicted is not None,
            f"Expected to find value for causal query {key}"
        )
        correct += 1
        total += 1
    
    print(f"CausalQuery: {correct}/{total} correct")

In [ ]:
# --------------------------------------------------------------------------------
# STEP 2: RUN ALL TASKS
# We use `kbench.llm` as a placeholder. This allows Kaggle to automatically swap
# in different models later when you use the "Add Models" button in the UI.
# --------------------------------------------------------------------------------

print("Running TemporalBench v2 Overlap tasks...")
print("=" * 60)

past_query_trap_task.run(kbench.llm)
print()
change_detection_task.run(kbench.llm)
print()
current_query_task.run(kbench.llm)
print()
causal_query_task.run(kbench.llm)

print()
print("=" * 60)
print("All TemporalBench v2 tasks completed.")

# --------------------------------------------------------------------------------
# STEP 3: NEXT STEPS
# 1. Click "Save Task" (top right) to publish to the leaderboard.
# 2. Try %autopilot in a new cell to auto-generate tasks or write your own!
# --------------------------------------------------------------------------------